In [1]:
import sys; sys.path.append("..")
import pandas as pd
from pathlib import Path

frames = [pd.read_parquet(p) for p in sorted(Path("../data/processed/features").glob("*.parquet"))]
data = pd.concat(frames).sort_index()
print(data.shape,"| tickers:", data["ticker"].unique())

(12000, 14) | tickers: <ArrowStringArray>
['AAPL', 'GOOGL', 'JNJ', 'JPM', 'META', 'MSFT', 'NVDA', 'TSLA', 'XOM', 'AMZN']
Length: 10, dtype: str


In [2]:
FEATURES = ["ret_1d","ret_2d","ret_3d","ret_5d","ret_10d",
            "price_vs_ma20","ma20_vs_ma50","vol_20d","rsi_14","macd","volume_z"]
train= data[data.index < "2024-01-01"]
test = data[data.index >= "2025-01-01"]
X_train, y_train = train[FEATURES], train["target_1d"].astype(int)
X_test,  y_test  = test[FEATURES],  test["target_1d"].astype(int)
print(len(X_train), "train rows |", len(X_test), "test rows")

5670 train rows | 3810 test rows


In [3]:
baseline = y_test.mean()
print(f"ALWAYS-UP BASELINE on test: {max(baseline, 1-baseline):.3f}")

ALWAYS-UP BASELINE on test: 0.531


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

models= {
    "logreg": Pipeline([("Scaler", StandardScaler()),
                        ("clf", LogisticRegression(max_iter=1000))]),
    "rf": RandomForestClassifier(n_estimators=300,max_depth=5, random_state=42,n_jobs=-1),
    "xgb": XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                         random_state=42, eval_metric="logloss"),
}

results={}
for name,model in models.items():
    model.fit(X_train,y_train)
    pred = model.predict(X_test)
    proba= model.predict_proba(X_test)[:,1]
    results[name] = {
        "accuracy":  accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall":    recall_score(y_test, pred),
        "f1":        f1_score(y_test, pred),
        "roc_auc":   roc_auc_score(y_test, proba),
    }

pd.DataFrame(results).T.round(3)

,accuracy,precision,recall,f1,roc_auc
logreg,0.529,0.536,0.838,0.654,0.505
rf,0.523,0.534,0.797,0.639,0.502
xgb,0.510,0.535,0.606,0.568,0.510


In [9]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.base import clone
import numpy as np

tss = TimeSeriesSplit(n_splits=5)
tv = data[data.index < "2025-01-01"].sort_index()
X_tv, y_tv = tv[FEATURES], tv["target_1d"].astype(int)

for name in ["logreg", "xgb"]:
    scores = []
    for tr_idx, te_idx in tss.split(X_tv):
        m = clone(models[name])
        m.fit(X_tv.iloc[tr_idx], y_tv.iloc[tr_idx])
        scores.append(accuracy_score(y_tv.iloc[te_idx], m.predict(X_tv.iloc[te_idx])))
    print(f"{name}: {np.mean(scores):.3f} ± {np.std(scores):.3f}   folds: {[round(s,3) for s in scores]}")

logreg: 0.502 ± 0.019   folds: [0.478, 0.499, 0.487, 0.516, 0.528]
xgb: 0.501 ± 0.014   folds: [0.475, 0.511, 0.503, 0.501, 0.514]


In [10]:
import plotly.express as px
imp = pd.Series(models["xgb"].feature_importances_, index=FEATURES).sort_values()
px.bar(imp, orientation="h", title="XGBoost: which features mattered?")

## Day 9 verdict
- Baseline (always-up): **0.531**. Best model: logreg at 0.529 — **nobody beat the coin.**
- ROC-AUC ~0.50-0.51 → no ranking skill: our 11 features carry ~no next-day signal for 2025.
- Walk-forward: logreg **0.502 ± 0.019**, xgb **0.501 ± 0.014** → statistically identical →
  chose **LogReg** (Occam's razor: among models without signal, ship the simplest).
- This null result is the honest, expected outcome for daily direction from public price
  features (efficient markets; Renaissance ran at ~50.75%). A leaky pipeline would have
  "won" here — ours refused to lie. Next candidates for real signal: sentiment features
  (Day 12), longer horizons, volatility prediction instead of direction.
